# Notebook 02e — PPO-CF on `MiniGrid-UnlockPickup-v0`

Three arms from **one** config, differing in exactly one field, `ppo.pg_mode`:

| arm | `pg_mode` | what it is |
|---|---|---|
| `gae` | `gae` | plain PPO — the control |
| `cf` | `cf_all_action` | PPO-CF, the full direct oracle |
| `shuf` | `cf_shuffled` | the same oracle vectors on **different** states |

The `shuf` arm is the gate, not an afterthought: if permuting the oracle across
states reproduces the benefit, the benefit was never state-local counterfactual
credit.

**Why UnlockPickup.** The working baseline needs a fixed layout, a probability
floor, and high entropy (`ent_coef=0.1`) to keep the key/door/box sequence alive.
That makes it a better PPO-CF test than a task solved by incidental reward: the
sampled GAE signal is sparse and noisy, but the oracle can evaluate every action
from the same restored state.

**What must be watched.** The sub-goal ladder is key → door → success. The probe
logs the first two rungs: `subgoal1` is carrying the key and `subgoal2` is door
open. Success is later, when the target box is picked up. If `subgoal2` rises but
success stays flat, inspect `pickup`, `drop`, and `toggle` before changing the
algorithm.

**Why H=64 initially.** Unlock used `cf_horizon=32` because door opening was the
terminal reward. UnlockPickup's reward is one leg later, so this notebook starts
at H=64. After the oracle check reports `cf_reward_coverage`, try H=32 if H=64
is expensive and coverage is already healthy.


---
## ▶ Knobs

In [ ]:
# ----------------------------------------------------------------------------
# EDIT ME
# ----------------------------------------------------------------------------
ENV_CONFIG    = "unlockpickup_cf"
SEEDS         = [0]               # None -> use the config's seeds
FORCE_RETRAIN = True

RUN_ORACLE_CHECKS = True          # run after the baseline has trajectories.npz
RUN_CONTROL       = True          # arm A: plain PPO
RUN_CF            = True          # arm B: PPO-CF
RUN_SHUFFLED      = False         # arm C: shuffled-oracle control (the P1 gate)

# subgoal1 / subgoal2 in THIS env. Success is the third rung: target box pickup.
SUBGOAL_LABELS = ("Picked up the key", "Opened the door")

# The oracle checks are meaningful only against a policy that does something.
# This is the fixed-layout baseline from 01_ppo_baseline_unlockpickup. It must
# have FINISHED for trajectories.npz to exist; otherwise section 1 falls back to
# a live rollout and should be treated as a smoke check, not evidence.
ORACLE_CHECK_RUN  = "unlockpickup_fixed0_floor"
ORACLE_CHECK_CKPT = 0.75

OVERRIDES = {
    # "ppo.total_timesteps": 200_000,   # a smoke run before the full 1M
    # "ppo.cf_horizon": 32,             # cheaper; use if H=64 coverage is good
    # "ppo.cf_subsample": 0.10,         # doubles labelled states and cost
    # "ppo.alpha_cf": 0.1,              # CF as a smaller correction
    # "ppo.target_kl": 0.03,            # compare arms with a KL cap if CF steps run hot
}
# ----------------------------------------------------------------------------


In [ ]:
# Reload edited modules automatically. Note this still cannot add a field to
# an already-imported dataclass -- for config schema changes, restart the kernel.
%load_ext autoreload
%autoreload 2

import sys, pathlib, time

ROOT = pathlib.Path.cwd()
if not (ROOT / "config").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import make_config, seed_dir, RUNS_DIR, FIGURES_DIR
from dataio import load_trajectories, load_checkpoint, list_checkpoints
from utils.logging import read_scalars
from utils.plotting import plot_subgoal_ladder, savefig
from scripts.train import run_seeds

base = make_config(ENV_CONFIG, **OVERRIDES)
if SEEDS is not None:
    base = make_config(ENV_CONFIG, **{**OVERRIDES, "run.seeds": tuple(SEEDS)})
SEEDS = tuple(base.run.seeds)
STEM  = base.run.run_name

# ONE config, two arms. Every hyperparameter is shared by construction; the only
# difference between the runs is the policy gradient.
# Three arms, per the plan's P1: PPO, PPO+oracle, and PPO+SHUFFLED oracle.
# The shuffled arm gets the same oracle vectors assigned to different batch
# states. It is the gate: if it reproduces the benefit, the benefit was never
# counterfactual information, just the extra gradient structure.
ARMS = {
    k: make_config(ENV_CONFIG, **{**OVERRIDES, "run.seeds": SEEDS,
                                  "ppo.pg_mode": m, "run.run_name": f"{STEM}_{k}"})
    for k, m in [("gae", "gae"), ("cf", "cf_all_action"), ("shuf", "cf_shuffled")]
}
FIG = FIGURES_DIR / f"nb02_{STEM}"
FIG.mkdir(parents=True, exist_ok=True)

print(ARMS["cf"].summary())
print(f"\narms: {[c.run.run_name for c in ARMS.values()]}   seeds: {list(SEEDS)}")

# Order-of-magnitude only. UnlockPickup uses H=64, R=2, subsample 5%,
# so the CF arms are much slower than the plain PPO control. Read section 1.5
# for measured throughput on this machine before committing to all arms.
rate = {"gae": 1500, "cf": 120, "shuf": 120}
print("\nestimated runtime")
for k, c in ARMS.items():
    m = c.ppo.total_timesteps / rate[k] / 60
    print(f"  {k:3s}  {m:5.1f} min/seed  x {len(SEEDS)} seeds = {m*len(SEEDS):5.1f} min")

---
# 1. Test the oracle

Run this after the fixed-layout baseline has finished and written
`runs/unlockpickup_fixed0_floor/seed_0/trajectories.npz`. The first check proves
restore/replay exactness on recorded transitions; the targeted check then looks
specifically for a successful target-box pickup transition and replays it after
restore.


In [ ]:
from envs.env_pool import set_sim_state, get_sim_state
from oracle.online import (OnlineOracle, check_replay, check_centering,
                           check_restore_equivalence, landscape_summary)

cfg = ARMS["cf"]
ENV_KW = {"fully_observable": cfg.env.fully_observable} if cfg.env.env_id.startswith("MiniGrid") else {}
K = 7 if cfg.env.env_id.startswith("MiniGrid") else None

# States and a policy to check against.
sd = seed_dir(ORACLE_CHECK_RUN, SEEDS[0]) if ORACLE_CHECK_RUN else None
traj_path = sd / "trajectories.npz" if sd else None
if sd and traj_path.exists():
    traj = load_trajectories(traj_path)
    ckpts = list_checkpoints(sd / "checkpoints")
    if not ckpts:
        raise FileNotFoundError(f"no checkpoints found under {sd / 'checkpoints'}")
    loaded = [(p, load_checkpoint(p)) for p in ckpts]
    p_ck, ck = min(loaded, key=lambda pc: abs(pc[1].fraction - ORACLE_CHECK_CKPT))
    value_fn, probs_fn = ck.values, ck.probs
    src = f"{ORACLE_CHECK_RUN} @ {ck.fraction:.0%} ({p_ck.name})"
    N = min(600, len(traj))
    sims  = traj.sim_state[:N]
    acts  = traj.action[:N]
    rews  = traj.reward[:N]
    nsims = traj.next_sim_state[:N]
    K = traj.n_actions
else:
    from agents.ppo import PPOTrainer
    print(f"no completed trajectory at {traj_path}; using one live rollout for smoke-only oracle checks")
    _t = PPOTrainer(cfg, seed=SEEDS[0], progress=False)
    _t.pool.reset(); _t.collect_rollout()
    T, NE = cfg.ppo.n_steps, cfg.env.n_envs
    f = lambda a: a[:T].reshape((T*NE,) + a.shape[2:])
    sims, acts, rews, nsims = f(_t.buffer.sim_state), f(_t.buffer.actions), f(_t.buffer.rewards), f(_t.buffer.next_sim_state)
    value_fn, probs_fn = _t._values_np, _t._probs_np
    src = "fresh rollout"
    K = _t.pool.n_actions

oracle = OnlineOracle(cfg.env.env_id, K, cfg.ppo.gamma, ENV_KW,
                      cfg.env.max_episode_steps, restore=cfg.ppo.cf_restore)
print(f"oracle check source: {src}; {len(sims)} states; K={K}")


### 1.1 Restore and replay must be exact

Restore each recorded simulator state, replay the action that was actually
taken, and compare against what the run recorded. **This is the check that
matters.** If it fails, every $A_{CF}$ in the project is meaningless and every
downstream number will look plausible and be wrong.

In [ ]:
rep = check_replay(oracle, sims[:400], acts[:400], rews[:400], nsims[:400])
print(f"  transitions replayed  {rep['n']}")
print(f"  max reward error      {rep['max_reward_error']:.3e}")
print(f"  max state error       {rep['max_state_error']:.3e}")
print(f"  -> {'EXACT' if rep['exact'] else 'MISMATCH — STOP, do not train'}")

### 1.2 A successful box pickup must survive state restore

The rare transition that matters in this environment is the terminal `pickup` of
the target box. The generic replay check can miss this if the source trajectory
has few successes, so this cell searches the baseline trajectory for a positive
reward pickup and replays exactly that transition after restore.


In [ ]:
success_idx = np.flatnonzero((acts.astype(int) == 3) & (rews > 0))
if len(success_idx) == 0:
    target_pickup_ok = False
    print("  no successful target-box pickup transition found in the oracle-check source")
    print("  finish a successful baseline run first; otherwise this check cannot validate the reward transition")
else:
    from envs.env_pool import make_env as _make
    matches = 0
    for j, idx in enumerate(success_idx[:20]):
        st = sims[idx]
        e = _make(cfg.env.env_id, cfg.env.max_episode_steps, **ENV_KW)
        set_sim_state(e, st, elapsed_steps=int(st[-1]))
        _, r_res, t_res, _, _ = e.step(3)
        ok = abs(float(rews[idx]) - float(r_res)) < 1e-9 and bool(t_res)
        matches += int(ok)
        if j < 5:
            print(f"  idx {idx}: recorded reward {float(rews[idx]):.4f}; restored reward {float(r_res):.4f}; terminated {t_res}")
        e.close()
    target_pickup_ok = matches == min(len(success_idx), 20)
    print()
    print(f"  successful pickup transitions replayed after restore: {matches}/{min(len(success_idx), 20)}")
    print(f"  -> {'OK' if target_pickup_ok else 'BROKEN - target pickup reward did not reproduce'}")


### 1.3 Policy centering

$\sum_a \pi(a|s) A_{CF}(s,a) = 0$ must hold to floating-point precision, per
state. It is what makes the all-action gradient unbiased with respect to action
sampling; if it drifts, the loss acquires a state-dependent bias term and the
whole construction is unsound. The trainer logs this every update as
`cf_centering` and it should stay at zero for the entire run.

In [ ]:
# Recover the observation in each state the same way training does, then pi(.|s).
obs0 = np.stack([set_sim_state(oracle.env, s, elapsed_steps=int(s[-1])) for s in sims[:300]])
pi = probs_fn(obs0)
print(f"behaviour policy recovered for {len(pi)} states; centering is checked in 1.4")


### 1.4 Is the landscape non-degenerate, and does it say sensible things?

Two different questions. *Non-degenerate*: do the actions actually differ, or is
$A_{CF}$ flat? *Sensible*: `pickup`, `drop`, and `toggle` are the actions that
can advance this task, while `done` is useless. `drop` is not marked useless here
because opening the locked door does not consume the key in upstream MiniGrid,
and the target box can only be picked up with an empty carrying slot.


In [ ]:
# Eq (3): branch, FORCE a, follow pi for cf_horizon steps, average over
# cf_rollouts continuations with common random numbers. NOT r + gamma*V(s').
a_cf, q_cf, diag = oracle.a_g(
    sims[:300], pi[:300], probs_fn, value_fn,
    horizon=cfg.ppo.cf_horizon, n_rollouts=cfg.ppo.cf_rollouts,
    bootstrap_tail=cfg.ppo.cf_bootstrap_tail, chunk=cfg.ppo.cf_branch_envs)
c = check_centering(a_cf, pi[:300])

print("INFORMATIVENESS -- does Q_g contain any actual reward?")
print(f"  branches that saw reward         {diag['reward_coverage']:.3f}")
print(f"  branches terminating within H    {diag['terminated_within_horizon']:.3f}")
print(f"  states with reward in some branch {diag['frac_states_any_reward']:.3f}")
print("  if this is near zero, raise cf_horizon or check that the baseline source actually solves")

s_ = landscape_summary(a_cf, q_cf, pi[:300], spread_threshold=cfg.oracle.spread_threshold,
                       useless_actions=(6,) if K == 7 else ())
names = ["left", "right", "forward", "pickup", "drop", "toggle", "done"][:K]
print()
print("SHAPE OF THE LANDSCAPE")
print(f"  centering, max |sum pi*A_CF|     {c['max_abs']:.3e}  -> {'OK' if c['ok'] else 'BIASED'}")
print(f"  mean |A_CF|                      {s_['mean_abs_a_cf']:.4f}")
print(f"  frac states with spread > {cfg.oracle.spread_threshold}    {s_['frac_states_with_spread']:.3f}")
print(f"  all finite                       {s_['finite']}")
print()
print("  best action by A_CF:")
for n, c_ in zip(names, s_["best_action_counts"]):
    tag = "   <-- useless action" if n == "done" else ""
    print(f"    {n:8s} {c_:5d}{tag}")
if "useless_actions_negative_frac" in s_:
    print()
    print(f"  done has A_CF < 0 in {s_['useless_actions_negative_frac']:.1%} of states")


### 1.5 `fast` restore vs `exact`, and throughput

`cf_restore: "exact"` goes through `set_sim_state`, which calls `env.reset()`
before every restore — that reset is most of the cost. `"fast"` writes the grid
and agent state directly. It is only worth having if it is **bit-identical**,
so verify rather than assume, then read the timing and decide.

In [ ]:
eq = check_restore_equivalence(cfg.env.env_id, K, cfg.ppo.gamma, sims[:250],
                               ENV_KW, cfg.env.max_episode_steps)
print(f"  max reward diff   {eq['max_reward_diff']:.3e}")
print(f"  max obs diff      {eq['max_obs_diff']:.3e}")
print(f"  terminated agree  {eq['terminated_agreement']:.3f}")
print(f"  -> {'IDENTICAL — fast is safe to use' if eq['identical'] else 'DIFFERENT — keep exact'}\n")

batch = cfg.env.n_envs * cfg.ppo.n_steps
for kind in ("exact", "fast"):
    o = OnlineOracle(cfg.env.env_id, K, cfg.ppo.gamma, ENV_KW, cfg.env.max_episode_steps, restore=kind)
    probe = sims[:min(batch, len(sims))]
    t0 = time.time(); o.q_cf(probe, value_fn); dt = time.time() - t0
    sps = len(probe) / dt
    print(f"  {kind:5s}  {sps:6.0f} collected steps/s   "
          f"-> {cfg.ppo.total_timesteps/sps/60:5.1f} min/seed at {cfg.ppo.total_timesteps:,} frames")
    o.close()
oracle.close()

### Oracle verdict

In [ ]:
checks = {
    "restore/replay is exact":                    rep["exact"],
    "A_CF is policy-centred (max < 1e-4)":        c["ok"],
    "A_CF is finite":                             s_["finite"],
    f"actions differ in > {cfg.oracle.min_frac_states_with_spread:.0%} of states":
        s_["frac_states_with_spread"] > cfg.oracle.min_frac_states_with_spread,
    "fast and exact restore agree":               eq["identical"],
    "successful target pickup survives restore":  target_pickup_ok,
    # Exactness says the oracle computes what it claims; this says what it
    # computes carries reward information for this task.
    "Q_g contains reward (coverage > 0.02)":      diag["reward_coverage"] > 0.02,
}
w = max(len(k) for k in checks)
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k:<{w}}")
print()
print("ORACLE:", "PASS — safe to train on" if all(checks.values())
      else "FAIL — fix before trusting CF results; A_CF errors are invisible downstream")


---
# 2. Arm A — plain PPO (control)

In [ ]:
def train_arm(key):
    c = ARMS[key]
    done = all((seed_dir(c.run.run_name, s) / "scalars.csv").exists() for s in c.run.seeds)
    if done and not FORCE_RETRAIN:
        print(f"found existing run at {RUNS_DIR / c.run.run_name} — skipping")
        return None
    t0 = time.time()
    out = run_seeds(c)
    print(f"\n{key}: {(time.time()-t0)/60:.1f} min total")
    return out

if RUN_CONTROL:
    train_arm("gae")


---
# 3. Arm B — PPO-CF

The first rollout prints the oracle's replay and centering check again, this
time against the live policy inside the trainer. `cf_centering` in
`scalars.csv` should read 0 on every logged update.

In [ ]:
if RUN_CF:
    train_arm("cf")

# The P1 control arm. Same oracle vectors, wrong states.
if RUN_SHUFFLED:
    train_arm("shuf")


---
# 4. Comparison

The metric is **sample efficiency**: frames to first reach
`success_rate_100 >= 0.9`. Also read `subgoal2_rate_100`: if both arms open the
door but only one reaches success, the difference is in the final box-pickup leg,
not in key-door credit assignment.


In [ ]:
scal = {k: {s: read_scalars(seed_dir(c.run.run_name, s) / "scalars.csv")
            for s in c.run.seeds if (seed_dir(c.run.run_name, s) / "scalars.csv").exists()}
        for k, c in ARMS.items()}
scal = {k: v for k, v in scal.items() if v}

THRESH = 0.9
def frames_to(d, col="success_rate_100", thresh=THRESH):
    hit = d.loc[d[col] >= thresh, "global_step"]
    return int(hit.iloc[0]) if len(hit) else None

rows = []
for arm, per_seed in scal.items():
    for s, d in per_seed.items():
        rows.append({
            "arm": arm, "seed": s,
            f"frames_to_{THRESH:g}": frames_to(d),
            "final_success": round(float(d["success_rate_100"].iloc[-1]), 3),
            "final_subgoal2": round(float(d["subgoal2_rate_100"].iloc[-1]), 3),
            "best_success": round(float(d["success_rate_100"].max()), 3),
            "ev_median": round(float(d["explained_variance"].median()), 3),
            "kl_median": round(float(d["approx_kl"].median()), 4),
            "clipfrac_median": round(float(d["clipfrac"].median()), 3),
            "cf_coverage": (round(float(d["cf_reward_coverage"].mean()), 3)
                            if "cf_reward_coverage" in d else None),
            "cf_corr_gae": (round(float(d["cf_corr_gae"].mean()), 3)
                            if "cf_corr_gae" in d else None),
        })
cmp_df = pd.DataFrame(rows).sort_values(["arm", "seed"])
display(cmp_df)

print(f"\nmedian frames to success >= {THRESH:g}")
for arm, g in cmp_df.groupby("arm"):
    v = g[f"frames_to_{THRESH:g}"].dropna()
    print(f"  {arm:3s}  {('%.0f' % v.median()) if len(v) else 'never':>10s}"
          f"   ({len(v)}/{len(g)} seeds reached it)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {"gae": "#3b6ea5", "cf": "#c4622d", "shuf": "#7a7a7a"}
panels = [("success_rate_100", "Success rate"), ("subgoal2_rate_100", SUBGOAL_LABELS[1]),
          ("entropy", "Policy entropy")]
for ax, (col, title) in zip(axes, panels):
    for arm, per_seed in scal.items():
        for i, (s, d) in enumerate(per_seed.items()):
            ax.plot(d["global_step"], d[col], lw=1.4, alpha=0.85, color=colors[arm],
                    label=arm if i == 0 else None)
    ax.set_title(title, fontsize=10); ax.set_xlabel("environment steps")
    ax.spines[["top","right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
axes[0].axhline(THRESH, color="#888", ls=":", lw=1)
axes[0].set_ylim(-0.05, 1.05); axes[1].set_ylim(-0.05, 1.05)
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout(); savefig(fig, FIG / "arms.png"); plt.show()

### The confound to check before believing a win

The all-action gradient updates **all K action logits** per state, while GAE
updates only the sampled one. At the same learning rate that can make the CF arm
take larger policy steps.

So if CF wins, the question is whether it won on counterfactual *information* or
just on a bigger effective step size. The panel below shows update size per arm.
If they are badly mismatched, rerun both with `ppo.target_kl: 0.03`, which caps
the per-update movement and makes the comparison KL-matched.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 3.8))
panels = [("approx_kl", "approx_kl (update size)"),
          ("clipfrac", "clipfrac"),
          ("adv_std_raw", "GAE advantage spread"),
          ("cf_reward_coverage", "Q_g reward coverage")]
for ax, (col, title) in zip(axes, panels):
    for arm, per_seed in scal.items():
        for i, (s, d) in enumerate(per_seed.items()):
            if col not in d.columns or not np.isfinite(d[col]).any():
                continue
            ax.plot(d["global_step"], d[col], lw=1.3, alpha=0.85, color=colors[arm],
                    label=arm if i == 0 else None)
    ax.set_title(title, fontsize=10); ax.set_xlabel("environment steps")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
axes[2].set_yscale("log")
axes[3].axhline(0.02, color="#888", ls=":", lw=1)
axes[3].text(0.02, 0.9, "minimum useful coverage", transform=axes[3].transAxes, fontsize=8)
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout(); savefig(fig, FIG / "update_size.png"); plt.show()

# PPO-CF health. Every column is checked before use: the trainer's cf_* keys
# change as the estimator changes, and a plot cell should not die on a rename.
for arm in ("cf", "shuf"):
    if arm not in scal:
        continue
    seed0 = sorted(scal[arm])[0]
    d = scal[arm][seed0]
    print()
    print(f"{ARMS[arm].run.run_name}  (seed {seed0})")
    checks = [
        ("cf_centering",       "max", "{:.3e}", "must stay ~0; above 1e-4 the all-action gradient is biased"),
        ("cf_reward_coverage", "last", "{:.3f}", "fraction of branches that saw reward; raise H if near zero"),
        ("cf_corr_gae",        "mean", "{:+.3f}", "corr(A_CF taken, GAE); read with coverage"),
        ("cf_mean_abs",        "mean", "{:.4f}", "|A_CF| in GAE-advantage units"),
        ("cf_n_cf_states",     "last", "{:.0f}", "states given a counterfactual per rollout"),
    ]
    for col, how, fmt, why in checks:
        if col not in d.columns or not np.isfinite(d[col]).any():
            print(f"  {col:20s} not logged")
            continue
        v = {"max": d[col].max(), "mean": d[col].mean(), "last": d[col].iloc[-1]}[how]
        print(f"  {col:20s} {how:>4} {fmt.format(v):>10}   {why}")


---
## 5. Reading the result

| outcome | what it means | next |
|---|---|---|
| CF reaches 0.9 in clearly fewer frames, KL comparable | the counterfactual advantage helps | expand the layout pool to `[0,1,2,3]` |
| CF wins but its `approx_kl` is much larger | probably step size, not information | rerun both with `ppo.target_kl: 0.03` |
| arms overlap | the fixed layout may be too easy to discriminate | run the shuffled arm, then expand the layout pool |
| CF is clearly worse | the oracle is exact, so suspect critic/bootstrap error | check `ev_median`, `cf_reward_coverage`, and target-pickup replay |

Once fixed layout is understood, the next comparison is the same config with
`env.layout_seeds: [0, 1, 2, 3]`. Keep the three arms identical except for
`ppo.pg_mode`; otherwise a win is not attributable to counterfactual credit.
